In [1]:
"""
Fake bank data generator for practising AML (Anti-Money Laundering) in SQL.

What this script does:
    1. Creates fake customers
    2. Creates accounts for those customers
    3. Creates normal (innocent) transactions
    4. Hides a few "crimes" inside the normal data
    5. Saves everything as CSV files in a folder called "data"

Run it with:   python generate_data.py

"""

import os
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from faker import Faker

# ----------------------------------------------------------------
# SETTINGS - change the numbers here if you want
# ----------------------------------------------------------------
random.seed(42)           # fixed seeds = same data every time we run
np.random.seed(42)
Faker.seed(42)
fake = Faker("en_IN")     # Indian-style names

NUM_CUSTOMERS = 5000
START_DATE = datetime(2025, 1, 1)
REPORTING_LIMIT = 1_000_000          # Rs 10 lakh. Cash above this is reported to the regulator
HIGH_RISK_COUNTRIES = ["Cayman Islands", "Panama", "Switzerland"]

# Average monthly income for each occupation
INCOME = {
    "Salaried": 60000,
    "Self-Employed": 80000,
    "Student": 15000,
    "Business Owner": 250000,
    "Retired": 30000,
    "Professional": 150000,
}
OCCUPATIONS = list(INCOME.keys())
OCCUPATION_CHANCES = [0.40, 0.15, 0.08, 0.12, 0.10, 0.15]   # must add up to 1

# Normal transaction types and how likely each one is
TXN_TYPES = ["UPI Payment", "Salary Credit", "ATM Withdrawal", "Cash Deposit",
             "Wire Transfer Out", "Wire Transfer In", "Crypto Purchase"]
TXN_CHANCES = [0.40, 0.12, 0.14, 0.10, 0.08, 0.08, 0.08]

# Countries money can be sent to, and how likely each one is
COUNTRIES = ["India", "Cayman Islands", "Panama", "Switzerland", "UAE", "USA", "UK"]
COUNTRY_CHANCES = [0.80, 0.01, 0.01, 0.01, 0.03, 0.07, 0.07]

In [2]:
# ----------------------------------------------------------------
# STEP 1: CREATE CUSTOMERS
# ----------------------------------------------------------------
customers = []
monthly_income = {}    # customer_id -> income (only used to size transactions later)

for customer_id in range(1, NUM_CUSTOMERS + 1):
    occupation = np.random.choice(OCCUPATIONS, p=OCCUPATION_CHANCES)

    customers.append({
        "Customer_ID": customer_id,
        "Full_Name": fake.name(),
        "Occupation": occupation,
        "Risk_Rating": np.random.choice(["Low", "Medium", "High"], p=[0.80, 0.15, 0.05]),
        "Country_of_Residence": np.random.choice(
            ["India", "UAE", "USA", "UK", "Singapore"], p=[0.94, 0.02, 0.02, 0.01, 0.01]),
        # PEP = Politically Exposed Person (very rare, only 0.5%)
        "Is_PEP": 1 if random.random() < 0.005 else 0,
    })

    # Everyone earns a bit different from the average for their job
    monthly_income[customer_id] = INCOME[occupation] * np.random.lognormal(0, 0.3)

customers_df = pd.DataFrame(customers)

In [3]:
# ----------------------------------------------------------------
# STEP 2: CREATE ACCOUNTS (every customer gets 1 or 2 accounts)
# ----------------------------------------------------------------
accounts = []
account_id = 100001    # first account number

for customer in customers:
    customer_id = customer["Customer_ID"]
    number_of_accounts = np.random.choice([1, 2], p=[0.7, 0.3])

    for _ in range(number_of_accounts):
        # Business owners always get a Business account.
        # Everyone else has a 3% chance of getting one.
        if customer["Occupation"] == "Business Owner" or random.random() < 0.03:
            account_type = "Business"
        else:
            account_type = np.random.choice(["Checking", "Savings"], p=[0.4, 0.6])

        # Account was opened somewhere between 30 days and 10 years before START_DATE
        days_before = random.randint(30, 3650)
        open_date = (START_DATE - timedelta(days=days_before)).date()

        accounts.append({
            "Account_ID": account_id,
            "Customer_ID": customer_id,
            "Account_Type": account_type,
            "Open_Date": open_date,
            "Current_Balance": round(monthly_income[customer_id] * np.random.lognormal(1, 0.8), 2),
        })
        account_id += 1

accounts_df = pd.DataFrame(accounts)

# Handy lists we will need later
all_account_ids = list(accounts_df["Account_ID"])
business_account_ids = list(accounts_df[accounts_df["Account_Type"] == "Business"]["Account_ID"])


In [4]:
# ----------------------------------------------------------------
# STEP 3: CREATE NORMAL TRANSACTIONS
# ----------------------------------------------------------------
# All transactions (normal + crime) are stored in this one list.
# Each transaction is a small dictionary = one row of the table.
transactions = []


def add_transaction(account_id, date, txn_type, amount,
                    country="India", channel="In-Branch", counterparty=None):
    """Adds one transaction (one row) to the transactions list."""
    transactions.append({
        "Account_ID": account_id,
        "Transaction_Date": date,
        "Transaction_Type": txn_type,
        "Amount": round(amount, 2),
        "Destination_Country": country,
        "Channel": channel,
        "Counterparty_Account_ID": counterparty,
    })


for account in accounts:
    income = monthly_income[account["Customer_ID"]]

    # Business accounts are busier, so they get more transactions
    if account["Account_Type"] == "Business":
        average_count = 180
    else:
        average_count = 90
    number_of_txns = np.random.poisson(average_count)    # random number around the average

    for _ in range(number_of_txns):
        txn_type = np.random.choice(TXN_TYPES, p=TXN_CHANCES)

        # Amount: lognormal gives many small amounts and a few big ones, like real life.
        # A typical transaction is around 4% of the person's monthly income.
        amount = np.random.lognormal(mean=np.log(income * 0.04), sigma=0.9)

        # Normal people don't deposit or withdraw huge cash,
        # so we cap those. This keeps normal data separate from the crimes.
        if txn_type == "Cash Deposit":
            amount = min(amount, 150_000)
        elif txn_type == "ATM Withdrawal":
            amount = min(amount, 40_000)

        # Date: a random day in the year
        date = START_DATE + timedelta(days=random.randint(0, 364))

        # Country: India, unless money is leaving (wire out / crypto)
        country = "India"
        if txn_type in ["Wire Transfer Out", "Crypto Purchase"]:
            country = np.random.choice(COUNTRIES, p=COUNTRY_CHANCES)

        # Channel: cash deposits always happen in the branch, others are random
        if txn_type == "Cash Deposit":
            channel = "In-Branch"
        else:
            channel = np.random.choice(["Mobile", "Online", "In-Branch"], p=[0.5, 0.3, 0.2])

        # Wire transfers involve another account (the "counterparty"), pick one at random
        counterparty = None
        if txn_type.startswith("Wire"):
            counterparty = int(np.random.choice(all_account_ids))

        add_transaction(account["Account_ID"], date, txn_type, amount, country, channel, counterparty)

# Remember how many transactions are normal (everything after this is a crime or decoy)
normal_count = len(transactions)

In [5]:
# ----------------------------------------------------------------
# STEP 4: HIDE THE CRIMES
# ----------------------------------------------------------------
ground_truth = []    # the answer key: which account is guilty of which crime
case_number = 0


def random_date():
    """Picks a random day in the year (not too close to the start or end)."""
    return START_DATE + timedelta(days=random.randint(5, 349))


# Shuffle the accounts, then hand out slices so no account is used for two crimes
shuffled = list(all_account_ids)
random.shuffle(shuffled)

structuring_accounts = shuffled[0:60]       # 60 accounts
rapid_accounts = shuffled[60:100]           # 40 accounts
roundtrip_accounts = shuffled[100:175]      # 75 accounts = 25 rings of 3

# Decoys: business accounts that are NOT used for any crime
used_accounts = set(shuffled[0:175])
decoy_accounts = [a for a in business_account_ids if a not in used_accounts][:30]


# --- Crime 1: STRUCTURING ---------------------------------------
# Splitting a big cash amount into several deposits just under the
# Rs 10 lakh limit, so the automatic report is never triggered.
for account_id in structuring_accounts:
    case_number += 1
    first_day = random_date()

    for _ in range(random.randint(3, 5)):                      # 3 to 5 deposits
        day = first_day + timedelta(days=random.randint(0, 2))  # all within 3 days
        amount = random.uniform(0.80, 0.99) * REPORTING_LIMIT   # 80% to 99% of the limit
        add_transaction(account_id, day, "Cash Deposit", amount)

    ground_truth.append((account_id, "Structuring", case_number))


# --- Crime 2: RAPID MOVEMENT OF FUNDS ----------------------------
# A big amount comes in and almost all of it leaves within a couple
# of days, going to a high-risk country.
for account_id in rapid_accounts:
    case_number += 1
    day_in = random_date()
    day_out = day_in + timedelta(days=random.randint(0, 2))

    amount_in = random.uniform(20, 80) * 100_000          # Rs 20 to 80 lakh
    amount_out = amount_in * random.uniform(0.80, 0.95)   # 80% to 95% leaves
    country = random.choice(HIGH_RISK_COUNTRIES)

    add_transaction(account_id, day_in, "Wire Transfer In", amount_in,
                    channel="Online", counterparty=random.choice(all_account_ids))
    add_transaction(account_id, day_out, "Wire Transfer Out", amount_out,
                    country=country, channel="Online")

    ground_truth.append((account_id, "Rapid Movement", case_number))


# --- Crime 3: ROUND TRIPPING -------------------------------------
# Money goes A -> B -> C -> back to A, losing about 1% on each hop.
# This makes dirty money look like it came from a clean source.
for ring_number in range(25):
    case_number += 1

    # The 3 accounts in this ring
    start = ring_number * 3
    ring = roundtrip_accounts[start:start + 3]      # [A, B, C]

    first_day = random_date()
    amount = random.uniform(15, 60) * 100_000       # Rs 15 to 60 lakh

    for hop in range(3):
        sender = ring[hop]
        receiver = ring[(hop + 1) % 3]              # after C comes A again
        hop_amount = amount * (0.99 ** hop)         # loses ~1% each hop
        hop_day = first_day + timedelta(days=hop * 2)

        # Every hop is recorded twice: money leaving the sender...
        add_transaction(sender, hop_day, "Wire Transfer Out", hop_amount,
                        channel="Online", counterparty=receiver)
        # ...and the same money arriving at the receiver
        add_transaction(receiver, hop_day, "Wire Transfer In", hop_amount,
                        channel="Online", counterparty=sender)

    for account_id in ring:
        ground_truth.append((account_id, "Round Tripping", case_number))


# --- Decoys: INNOCENT look-alikes --------------------------------
# Honest businesses that make two big cash deposits. They look a bit
# suspicious but are NOT crimes, so they are not added to ground_truth.
# They help us check that our SQL rules are not too aggressive.
for account_id in decoy_accounts:
    first_day = random_date()
    second_day = first_day + timedelta(days=2)

    add_transaction(account_id, first_day, "Cash Deposit", random.uniform(4.5, 5.5) * 100_000)
    add_transaction(account_id, second_day, "Cash Deposit", random.uniform(4.5, 5.5) * 100_000)

In [6]:
# ----------------------------------------------------------------
# STEP 5: CLEAN UP AND SAVE
# ----------------------------------------------------------------
transactions_df = pd.DataFrame(transactions)

# Sort by date and give every transaction a unique ID (1, 2, 3, ...)
transactions_df = transactions_df.sort_values("Transaction_Date").reset_index(drop=True)
transactions_df["Transaction_ID"] = transactions_df.index + 1

# Keep only the date part (no time) and make the counterparty a whole number.
# ("Int64" allows whole numbers with blanks, otherwise we would see 123.0)
transactions_df["Transaction_Date"] = transactions_df["Transaction_Date"].dt.date
transactions_df["Counterparty_Account_ID"] = transactions_df["Counterparty_Account_ID"].astype("Int64")

# Put the columns in the same order as the SQL table
transactions_df = transactions_df[[
    "Transaction_ID", "Account_ID", "Transaction_Date", "Transaction_Type",
    "Amount", "Destination_Country", "Channel", "Counterparty_Account_ID",
]]

ground_truth_df = pd.DataFrame(ground_truth, columns=["Account_ID", "Typology", "Case_ID"])

# Save all 4 files
os.makedirs("data", exist_ok=True)
customers_df.to_csv("data/customers.csv", index=False)
accounts_df.to_csv("data/accounts.csv", index=False)
transactions_df.to_csv("data/transactions.csv", index=False)
ground_truth_df.to_csv("data/ground_truth.csv", index=False)

print("Customers:   ", len(customers_df))
print("Accounts:    ", len(accounts_df))
print("Transactions:", len(transactions_df))
print("Crime cases hidden:", case_number)

Customers:    5000
Accounts:     6463
Transactions: 666228
Crime cases hidden: 125
